# Experiment 6 — Class-wise Explanation Analysis

Research Contribution 4: investigate whether XAI explanations differ meaningfully
between healthy and diseased leaves, and across different disease types.

In [ ]:
import sys
sys.path.append('..')

import torch
import json
from pathlib import Path
import yaml

from backend.ml.models.resnet_model import load_checkpoint, get_device
from backend.ml.evaluation.classwise_analysis import (
    ClasswiseAnalyzer, attention_region_stats, scan_dataset_for_classwise
)
from backend.ml.evaluation.visualisations import (
    plot_classwise_heatmaps,
    plot_healthy_vs_diseased,
    plot_similarity_matrix,
)

with open('../configs/config.yaml') as f:
    CONFIG = yaml.safe_load(f)

DEVICE = get_device()

# Load model
CKPT = '../backend/ml/checkpoints/best_model.pth'
if Path(CKPT).exists():
    model, meta = load_checkpoint(CKPT, DEVICE)
    with open('../backend/ml/checkpoints/class_mapping.json') as f:
        IDX_TO_CLASS = {int(k): v for k, v in json.load(f)['idx_to_class'].items()}
else:
    from backend.ml.models.resnet_model import PlantDiseaseResNet
    model = PlantDiseaseResNet(num_classes=38, pretrained=True).to(DEVICE)
    IDX_TO_CLASS = {i: f'class_{i}' for i in range(38)}

print('Model ready.')

In [ ]:
# ── Scan dataset ──────────────────────────────────────
class_image_map = scan_dataset_for_classwise(
    CONFIG['data']['root'],
    images_per_class=20
)
print(f'Classes found: {len(class_image_map)}')

In [ ]:
# ── Compute per-class mean heatmaps ───────────────────
# This may take a few minutes (20 images × 38 classes)
analyzer = ClasswiseAnalyzer(
    model, DEVICE,
    image_size=224,
    images_per_class=20,
)

print('[Exp 6] Computing class-wise heatmaps...')
class_results = analyzer.compute_class_heatmaps(class_image_map)
print(f'\nDone. Analysed {len(class_results)} classes.')

In [ ]:
# ── Visualise mean heatmaps ───────────────────────────
plot_classwise_heatmaps(
    class_results,
    save_path='../outputs/evaluation/classwise_heatmaps.png',
    max_classes=12,
)

In [ ]:
# ── Healthy vs. diseased comparison ──────────────────
hvd = ClasswiseAnalyzer.healthy_vs_diseased(class_results)

print('Healthy vs. Diseased Statistics:')
print(f"  Healthy  — coverage: {hvd['healthy']['mean_coverage']:.3f} ± {hvd['healthy']['std_coverage']:.3f}")
print(f"  Diseased — coverage: {hvd['diseased']['mean_coverage']:.3f} ± {hvd['diseased']['std_coverage']:.3f}")
print(f"  Healthy  — consistency: {hvd['healthy']['mean_consistency']:.3f}")
print(f"  Diseased — consistency: {hvd['diseased']['mean_consistency']:.3f}")
print(f"\nInterpretation: {hvd['interpretation']}")

plot_healthy_vs_diseased(
    hvd,
    save_path='../outputs/evaluation/healthy_vs_diseased.png'
)

In [ ]:
# ── Class similarity matrix ───────────────────────────
sim_matrix, class_names = ClasswiseAnalyzer.class_similarity_matrix(class_results)

print(f'Similarity matrix shape: {sim_matrix.shape}')
print(f'Mean off-diagonal similarity: {sim_matrix[sim_matrix < 1].mean():.4f}')
print('  (High value = different classes rely on similar visual features)')

# Most similar pair
import numpy as np
n = len(class_names)
triu = np.triu_indices(n, k=1)
best_pair_idx = np.argmax(sim_matrix[triu])
i, j = triu[0][best_pair_idx], triu[1][best_pair_idx]
print(f'Most similar classes: {class_names[i]} ↔ {class_names[j]} = {sim_matrix[i,j]:.4f}')

plot_similarity_matrix(
    sim_matrix, class_names,
    save_path='../outputs/evaluation/class_similarity_matrix.png'
)

In [ ]:
# ── Attention region stats per class ─────────────────
print('\nAttention region statistics (sample classes):')
print(f'{"Class":<35} {"Center%":>8} {"Border%":>8} {"Entropy":>8} {"Coverage":>9}')
print('-' * 72)

for name, r in list(class_results.items())[:10]:
    stats = attention_region_stats(r['mean_heatmap'])
    print(
        f"{name[:35]:<35} "
        f"{stats['center_energy']*100:>7.1f}% "
        f"{stats['border_energy']*100:>7.1f}% "
        f"{stats['entropy']:>8.3f} "
        f"{r['coverage']:>9.3f}"
    )

In [ ]:
# ── Export all results ────────────────────────────────
import json, datetime

export = {
    'timestamp': datetime.datetime.now().isoformat(),
    'healthy_vs_diseased': {
        'healthy': hvd['healthy'],
        'diseased': hvd['diseased'],
    },
    'per_class': {
        name: {
            'coverage':    float(r['coverage']),
            'consistency': float(r['consistency']),
            'n_images':    r['n_images'],
            'peak_coords': list(r['peak_coords']),
        }
        for name, r in class_results.items()
    },
    'mean_cross_class_similarity': float(sim_matrix[np.triu_indices(n, k=1)].mean()),
}

out = '../outputs/evaluation/classwise_results.json'
with open(out, 'w') as f:
    json.dump(export, f, indent=2)

print(f'Results saved → {out}')